# Person A — Notebook 5: Sequential Editing under Localization × Subject-Overlap
**CS 590NN | Amogh | Apr 24 — novel-finding experiment**

## What this notebook is for

The single-edit comparison story (Notebook 4) is a corroboration of Guo et al. (2024). To get a defensible **novel** finding for the final report, we move to the regime nobody has cleanly studied: *sequential* edits under varying localization, where the second edit may or may not share a subject with the first.

## Hypothesis

When you edit fact A then fact B without restoring weights, the cross-edit interaction depends on *both* the localization signal and whether A and B share a subject. We predict:

1. **Subject-disjoint pairs**: all three localization methods (ACDC, ROME-trace, Random) preserve A and succeed on B. KL drift is roughly additive.
2. **Subject-shared pairs**: tighter circuits (ACDC at 1.5B, ROME-trace's top-K) overwrite the first edit *more* because the same components handle related factual recall. Random circuits with broader scope partially escape this.
3. **KL super-additivity** (`kl_AB - kl_A_solo - kl_B_solo`) is *positive* for shared-subject pairs and *near zero* for disjoint pairs.

Either outcome is a real result. A null result on the subject-overlap variable is itself novel — it would say localization choice doesn't survive sequential editing, extending Guo et al.'s single-edit finding.

## Speed design

- Reuse existing `week2_circuit_log_v2.1.json` and `week6_rome_trace_circuits.json` — no recomputation for the 50 baseline samples.
- Compute new circuits only for the additional samples needed for subject-shared pairing.
- Cache reference logits per sample, reuse across all 3 methods.
- Batched neutral-set KL computation (vs. one-prompt-at-a-time loop in Notebook 4).
- Per-sample try/except so one OOM doesn't kill the run.
- Reuses `run_edit_with_kl_polished` from Notebook 3 v2 — same protocol, same hyperparameters.

### 5.0 Install
Run once. Runtime restarts automatically — do not re-run after restart.

In [ ]:
import subprocess, sys, os

def pip(args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + args)

pip(["numpy==1.26.4"])
pip(["transformer-lens", "transformers", "datasets", "accelerate", "einops", "jaxtyping"])
pip(["matplotlib"])

print("Done. Restarting runtime...")
os.kill(os.getpid(), 9)

### 5.1 Imports — start here after restart

In [ ]:
import torch, json, requests, random, traceback
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
from collections import defaultdict, Counter
from datetime import datetime

from transformer_lens import HookedTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS_DIR = Path("results_nb5")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    free, total = torch.cuda.mem_get_info()
    print(f"GPU   : {torch.cuda.get_device_name(0)} ({free/1e9:.1f}/{total/1e9:.1f} GB free)")
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

### 5.2 Load Model

In [ ]:
MODEL_NAME_PRIMARY  = "Qwen/Qwen3-0.6B"
MODEL_NAME_FALLBACK = "Qwen/Qwen2.5-0.5B"

def load_model(name):
    return HookedTransformer.from_pretrained(
        name,
        center_unembed=True,
        center_writing_weights=False,
        fold_ln=True,
        refactor_factored_attn_matrices=False,
        device=DEVICE,
    )

try:
    model = load_model(MODEL_NAME_PRIMARY)
    MODEL_NAME = MODEL_NAME_PRIMARY
except Exception as e:
    print(f"Primary failed ({e}). Using fallback...")
    model = load_model(MODEL_NAME_FALLBACK)
    MODEL_NAME = MODEL_NAME_FALLBACK

model.set_use_attn_result(True)
model.eval()
model.tokenizer.pad_token = model.tokenizer.eos_token
free, _ = torch.cuda.mem_get_info()
print(f"Loaded : {MODEL_NAME}")
print(f"Config : {model.cfg.n_layers} layers | {model.cfg.n_heads} heads | d_model={model.cfg.d_model}")
print(f"Memory : {free/1e9:.1f} GB free after load")

### 5.3 Auto-Fetch Existing Circuit Logs from GitHub

Three files are pulled directly from your repo — no manual upload needed.
- `week2_circuit_log_v2.1.json` — ACDC circuits for the 50 base samples
- `week6_rome_trace_circuits.json` — ROME-trace top-K circuits for those samples
- `week5_harness_output_kl_v2.json` — solo-edit results at n_steps=5 (used as solo reference)

In [ ]:
import urllib.request, os

REPO_RAW = "https://raw.githubusercontent.com/rukmini-17/targeted-unlearning/main"
FILES = {
    "week2_circuit_log_v2.1.json":   f"{REPO_RAW}/circuit_pipeline/results/qwen_circuits/week2_circuit_log_v2.1.json",
    "week6_rome_trace_circuits.json": f"{REPO_RAW}/circuit_pipeline/results/qwen_comparison/week6_rome_trace_circuits.json",
    "week5_harness_output_kl_v2.json": f"{REPO_RAW}/circuit_pipeline/results/qwen_kl_ablation/week5_harness_output_kl_v2.json",
}
for fname, url in FILES.items():
    try:
        urllib.request.urlretrieve(url, fname)
        size_kb = os.path.getsize(fname) / 1024
        print(f"  fetched: {fname} ({size_kb:.1f} KB)")
    except Exception as e:
        print(f"  FAILED:  {fname} -- {e}")
        print(f"           URL was: {url}")

### 5.4 Load CounterFact and Build Pair Sets

Pair construction:
- **Subject-shared (P_share)**: same `subject`, different `relation_id`. Edits two facts about the same entity.
- **Subject-disjoint (P_disjoint)**: different subjects, length-matched on prompt token count.

Target: 15 of each, prioritizing samples that already have circuits computed.

In [ ]:
@dataclass
class EditSample:
    idx_cf:          int      # global index in CounterFact
    prompt:          str
    subject:         str
    relation_id:     str
    target_new:      str
    target_true:     str

print("Downloading CounterFact from ROME project source...")
raw = requests.get("https://rome.baulab.info/data/dsets/counterfact.json", timeout=120).json()
print(f"Downloaded {len(raw)} records")

def parse(i, rec):
    rr = rec["requested_rewrite"]
    return EditSample(
        idx_cf      = i,
        prompt      = rr["prompt"].format(rr["subject"]),
        subject     = rr["subject"],
        relation_id = rr["relation_id"],
        target_new  = " " + rr["target_new"]["str"],
        target_true = " " + rr["target_true"]["str"],
    )

# === Two-pass scan: index subjects FIRST without building full EditSample objects ===
# Avoids holding 5000 EditSample dataclasses in memory simultaneously.

SCAN_LIMIT = 5000

# Pass 1 — lightweight subject indexing (just reads strings, builds defaultdict)
subj_to_idx = defaultdict(list)
for i in range(min(SCAN_LIMIT, len(raw))):
    subj = raw[i]["requested_rewrite"]["subject"]
    subj_to_idx[subj].append(i)
shared_subjects = {s: idxs for s, idxs in subj_to_idx.items() if len(idxs) >= 2}
print(f"Subjects with 2+ facts in scanned window: {len(shared_subjects)}")

# Free the scan dict; we only need shared_subjects
del subj_to_idx
import gc; gc.collect()

In [ ]:
# Build subject-shared pairs from raw[] on demand (no full all_samples list)
N_PAIRS_TARGET = 15
share_pairs = []
for subj, idxs in shared_subjects.items():
    a_i, b_i = idxs[0], idxs[1]
    a, b = parse(a_i, raw[a_i]), parse(b_i, raw[b_i])
    if a.relation_id != b.relation_id:    # require different relations on same subject
        share_pairs.append((a, b))
    if len(share_pairs) >= N_PAIRS_TARGET:
        break
print(f"Subject-shared pairs built: {len(share_pairs)}")
for a, b in share_pairs[:3]:
    print(f"  [{a.idx_cf}] {a.prompt!r} ({a.relation_id}) -> {a.target_new.strip()!r}")
    print(f"  [{b.idx_cf}] {b.prompt!r} ({b.relation_id}) -> {b.target_new.strip()!r}\n")

In [ ]:
# Disjoint-pair construction without instantiating all 5000 samples upfront.
# Tokenize ON DEMAND for length matching.

def tok_len_from_raw(i):
    p = raw[i]["requested_rewrite"]
    prompt = p["prompt"].format(p["subject"])
    return len(model.to_tokens(prompt)[0])

share_lengths = [(tok_len_from_raw(a.idx_cf), tok_len_from_raw(b.idx_cf)) for a, b in share_pairs]
shared_subjects_set = set(shared_subjects.keys())

# Disjoint pool indices (NOT EditSample objects yet)
disjoint_pool_idx = [i for i in range(min(SCAN_LIMIT, len(raw)))
                    if raw[i]["requested_rewrite"]["subject"] not in shared_subjects_set]
random.shuffle(disjoint_pool_idx)

# Bucket by token length lazily
by_len = defaultdict(list)
# Cap how many we tokenize (memory) — 1500 is plenty to find length matches
for i in disjoint_pool_idx[:1500]:
    by_len[tok_len_from_raw(i)].append(i)

disjoint_pairs = []
used = set()
for La, Lb in share_lengths:
    a_cands = [i for i in by_len.get(La, []) if i not in used]
    b_cands = [i for i in by_len.get(Lb, []) if i not in used]
    if not a_cands or not b_cands:
        a_cands = [i for L in (La, La-1, La+1) for i in by_len.get(L, []) if i not in used]
        b_cands = [i for L in (Lb, Lb-1, Lb+1) for i in by_len.get(L, []) if i not in used]
    if a_cands and b_cands:
        ai, bi = a_cands[0], b_cands[0]
        a, b = parse(ai, raw[ai]), parse(bi, raw[bi])
        if a.idx_cf != b.idx_cf and a.subject != b.subject:
            disjoint_pairs.append((a, b))
            used.add(ai); used.add(bi)
    if len(disjoint_pairs) >= N_PAIRS_TARGET: break

print(f"Subject-disjoint pairs built: {len(disjoint_pairs)}")
for a, b in disjoint_pairs[:3]:
    print(f"  [{a.idx_cf}] {a.prompt!r} -> {a.target_new.strip()!r}")
    print(f"  [{b.idx_cf}] {b.prompt!r} -> {b.target_new.strip()!r}\n")

# Collect distinct samples needing circuits
needed_samples = {}
for a, b in share_pairs + disjoint_pairs:
    needed_samples[a.idx_cf] = a
    needed_samples[b.idx_cf] = b
print(f"\nDistinct samples needing circuits: {len(needed_samples)}")

# Free the large raw list and disjoint pool — we have what we need
del by_len, disjoint_pool_idx
gc.collect()

### 5.5 Load Existing Circuit Logs (and Identify Missing Ones)

Reuses any sample whose `idx_cf` already has a circuit in the uploaded logs (the original 50-sample pool from Notebooks 1/4 used CounterFact indices 0..49). New samples will be discovered fresh in 5.6.

In [ ]:
def load_if_present(fname):
    p = Path(fname)
    if not p.exists(): return None
    with open(p) as f: return json.load(f)

acdc_log_v2 = load_if_present("week2_circuit_log_v2.1.json") or []
rome_trace_log = load_if_present("week6_rome_trace_circuits.json") or []
solo_harness = load_if_present("week5_harness_output_kl_v2.json") or {}

print(f"Loaded ACDC entries        : {len(acdc_log_v2)}")
print(f"Loaded ROME-trace entries  : {len(rome_trace_log)}")
print(f"Loaded solo harness ablations: {len(solo_harness)} step-counts")

# Map by idx (these were indexed by ORDER in the prior runs, which corresponds to CounterFact idx 0..49)
acdc_by_cf_idx = {e["idx"]: e for e in acdc_log_v2}
rome_by_cf_idx = {e["idx"]: e for e in rome_trace_log}

# Identify missing
missing_acdc = [idx for idx in needed_samples if idx not in acdc_by_cf_idx]
missing_rome = [idx for idx in needed_samples if idx not in rome_by_cf_idx]
print(f"\nMissing ACDC circuits   : {len(missing_acdc)} samples")
print(f"Missing ROME-trace circuits: {len(missing_rome)} samples")

### 5.6 Compute Missing Circuits

Reuses the v2.1 ACDC and ROME-trace implementations (same code path as Notebooks 1 and 4 — just compacted).

In [ ]:
# --- ACDC (single-pass activation patching with threshold) ---
class NativeACDC:
    def __init__(self, model, threshold=0.3, effect_floor=0.5):
        self.m, self.th, self.floor = model, threshold, effect_floor

    def _ld(self, logits, t, b):
        return (logits[0, -1, t] - logits[0, -1, b]).item()

    def run(self, prompt, target_true, target_new):
        m = self.m
        true_id = m.to_tokens(target_true, prepend_bos=False)[0,0].item()
        new_id  = m.to_tokens(target_new,  prepend_bos=False)[0,0].item()
        tokens  = m.to_tokens(prompt)
        corrupt = tokens.clone()
        if tokens.shape[1] > 2:
            corrupt[0, 1:-1] = torch.randint(1000, m.cfg.d_vocab-1, (tokens.shape[1]-2,))

        with torch.no_grad():
            cl, cc = m.run_with_cache(tokens)
            kl, kc = m.run_with_cache(corrupt)
        clean_score = self._ld(cl, true_id, new_id)
        corrupt_score = self._ld(kl, true_id, new_id)
        eff = max(abs(clean_score - corrupt_score), self.floor)

        attn_heads, mlp_layers = [], []
        for L in range(m.cfg.n_layers):
            hn = f"blocks.{L}.attn.hook_result"
            if hn in cc:
                for h in range(m.cfg.n_heads):
                    def mk(h=h, n=hn):
                        ca = kc[n][:,:,h:h+1,:].clone()
                        def fn(v, hook): v[:,:,h:h+1,:] = ca; return v
                        return fn
                    with torch.no_grad():
                        pl = m.run_with_hooks(tokens, fwd_hooks=[(hn, mk())])
                    if abs(self._ld(pl, true_id, new_id) - clean_score) / eff >= self.th:
                        attn_heads.append((L, h))
            hn = f"blocks.{L}.hook_mlp_out"
            if hn in cc:
                ca = cc[hn].clone()  # not used directly; we want corrupt patched in
                kca = kc[hn].clone()
                def mk_mlp(n=hn, kca=kca):
                    def fn(v, hook): return kca
                    return fn
                with torch.no_grad():
                    pl = m.run_with_hooks(tokens, fwd_hooks=[(hn, mk_mlp())])
                if abs(self._ld(pl, true_id, new_id) - clean_score) / eff >= self.th:
                    mlp_layers.append(L)
        del cc, kc, cl, kl
        torch.cuda.empty_cache()
        return {
            "attn_heads": sorted(set(attn_heads)),
            "mlp_layers": sorted(set(mlp_layers)),
            "n_attn": len(attn_heads), "n_mlp": len(mlp_layers),
        }

if missing_acdc:
    print(f"Computing ACDC circuits for {len(missing_acdc)} new samples...")
    acdc = NativeACDC(model, threshold=0.3, effect_floor=0.5)
    for idx in missing_acdc:
        s = needed_samples[idx]
        try:
            r = acdc.run(s.prompt, s.target_true, s.target_new)
            r["idx"] = idx; r["prompt"] = s.prompt
            acdc_by_cf_idx[idx] = r
            print(f"  [{idx}] n_attn={r['n_attn']:>3} n_mlp={r['n_mlp']:>2}")
        except Exception as e:
            print(f"  [{idx}] FAILED: {e}")
else:
    print("All ACDC circuits already cached — skipping.")

In [ ]:
# --- ROME-trace (causal tracing top-K MLPs) ---
NOISE_SIGMA = 0.1
TOP_K_TRACE = 5

def find_subject_range(model, prompt, subject):
    full = model.to_tokens(prompt)[0].tolist()
    pre  = prompt.find(subject)
    if pre < 0: return max(0, len(full)-3), len(full)
    pre_tok = model.to_tokens(prompt[:pre])[0].tolist()
    sub_tok = model.to_tokens(prompt[:pre+len(subject)])[0].tolist()
    return len(pre_tok), len(sub_tok)

def causal_trace(model, sample, n_noise=3):
    tokens = model.to_tokens(sample.prompt)
    new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0,0].item()
    s_start, s_end = find_subject_range(model, sample.prompt, sample.subject)
    with torch.no_grad():
        clean_logits, cache = model.run_with_cache(tokens)

    def noise_hook():
        def fn(emb, hook):
            emb[:, s_start:s_end, :] = emb[:, s_start:s_end, :] + torch.randn_like(emb[:, s_start:s_end, :]) * NOISE_SIGMA
            return emb
        return fn

    indirect = {}
    pos = s_end - 1
    for L in range(model.cfg.n_layers):
        hn = f"blocks.{L}.hook_mlp_out"
        clean_act = cache[hn][:, pos:pos+1, :].clone()
        def restore(act=clean_act, p=pos):
            def fn(v, hook): v[:, p:p+1, :] = act; return v
            return fn
        torch.manual_seed(42)
        prob_avg = 0.0
        for _ in range(n_noise):
            with torch.no_grad():
                rl = model.run_with_hooks(tokens, fwd_hooks=[
                    ("hook_embed", noise_hook()),
                    (hn, restore()),
                ])
            prob_avg += torch.softmax(rl[0,-1,:], dim=-1)[new_id].item()
        indirect[L] = prob_avg / n_noise
    del cache
    torch.cuda.empty_cache()
    top_k = sorted(indirect, key=lambda L: -indirect[L])[:TOP_K_TRACE]
    return {"mlp_layers": sorted(top_k), "attn_heads": [], "n_mlp": len(top_k), "n_attn": 0}

if missing_rome:
    print(f"Computing ROME-trace circuits for {len(missing_rome)} new samples...")
    for idx in missing_rome:
        s = needed_samples[idx]
        try:
            r = causal_trace(model, s)
            r["idx"] = idx
            rome_by_cf_idx[idx] = r
            print(f"  [{idx}] mlp_layers={r['mlp_layers']}")
        except Exception as e:
            print(f"  [{idx}] FAILED: {e}")
else:
    print("All ROME-trace circuits already cached — skipping.")

# Random circuits — size-matched to ACDC's n_mlp per sample (no attention)
random.seed(42)
random_by_cf_idx = {}
for idx in needed_samples:
    K = acdc_by_cf_idx[idx]["n_mlp"] if idx in acdc_by_cf_idx else 5
    layers = sorted(random.sample(range(model.cfg.n_layers), min(K, model.cfg.n_layers)))
    random_by_cf_idx[idx] = {"idx": idx, "mlp_layers": layers, "attn_heads": [], "n_mlp": len(layers), "n_attn": 0}
print(f"Random circuits built: {len(random_by_cf_idx)}")

### 5.7 Edit Functions (reuse Notebook 3 v2 protocol, with batched-KL speedup)

Two changes vs. Notebook 3 v2:
1. **`kl_loss_neutral_batched`** — pads all 32 neutral prompts to a single batch, runs ONE forward pass instead of 32. ~5-10× faster.
2. **No re-snapshot per sample** — original_state is reused across all pairs.

In [ ]:
NEUTRAL_PROMPTS = [
    "The sum of two and three is", "Twelve divided by four equals",
    "The square root of nine is", "Ten times ten equals",
    "The capital of Japan is", "The largest ocean on Earth is the",
    "Mount Everest is located in the", "The Amazon River flows through",
    "Water boils at one hundred degrees", "The chemical symbol for gold is",
    "Plants produce oxygen through a process called", "The Earth orbits the",
    "A week contains seven", "The primary colors are red, blue, and",
    "Humans have two lungs and one", "A triangle has three",
    "The opposite of hot is", "A baby cat is called a",
    "The past tense of run is", "A group of fish is called a",
    "The Industrial Revolution began in the eighteenth",
    "The Renaissance was a period of cultural",
    "World War Two ended in the year", "The Wright Brothers invented the",
    "A computer's main processor is the", "The world wide web was invented by",
    "An algorithm is a sequence of", "The unit of electrical resistance is the",
    "Roses are typically red while violets are",
    "The sky appears blue because of light",
    "A year contains twelve", "The freezing point of water is zero degrees",
]
assert len(NEUTRAL_PROMPTS) == 32

In [ ]:
def cache_reference_logits_batched(model, prompts):
    """Pad prompts to common length, run ONE forward pass, return (tokens, last_pos, ref_lp)."""
    tokenized = [model.to_tokens(p)[0] for p in prompts]
    max_len = max(t.shape[0] for t in tokenized)
    pad_id = model.tokenizer.pad_token_id
    padded = torch.full((len(prompts), max_len), pad_id, dtype=tokenized[0].dtype, device=DEVICE)
    last_pos = torch.zeros(len(prompts), dtype=torch.long, device=DEVICE)
    for i, t in enumerate(tokenized):
        padded[i, :t.shape[0]] = t.to(DEVICE)
        last_pos[i] = t.shape[0] - 1
    model.eval()
    with torch.no_grad():
        logits = model(padded)                              # [B, L, V]
    last_logits = logits[torch.arange(len(prompts)), last_pos, :]   # [B, V]
    ref_lp = torch.log_softmax(last_logits, dim=-1).detach().clone()  # [B, V]
    del logits, last_logits
    return padded, last_pos, ref_lp

def kl_loss_neutral_batched(model, padded, last_pos, ref_lp, train=True):
    """Mean KL(p_ref || p_edit) across the batch in ONE forward pass."""
    if train:
        logits = model(padded)
    else:
        with torch.no_grad():
            logits = model(padded)
    last_logits = logits[torch.arange(padded.shape[0]), last_pos, :]
    edit_lp = torch.log_softmax(last_logits, dim=-1)
    p_ref = ref_lp.exp()
    kl_per_prompt = (p_ref * (ref_lp - edit_lp)).sum(dim=-1)   # [B]
    return kl_per_prompt.mean()

def restore_weights(model, state):
    with torch.no_grad():
        for name, param in model.named_parameters():
            param.copy_(state[name])
    torch.cuda.empty_cache()

def get_circuit_params(model, attn, mlp):
    seen, params = set(), []
    # Dedup attention W_O references (one W_O per layer regardless of head count)
    for (L, h) in attn:
        if ("attn_W_O", L) in seen: continue
        try:
            params.append(model.blocks[L].attn.W_O); seen.add(("attn_W_O", L))
        except Exception: pass
    for L in mlp:
        if ("mlp", L) in seen: continue
        try:
            params.append(model.blocks[L].mlp.W_in); params.append(model.blocks[L].mlp.W_out)
            seen.add(("mlp", L))
        except Exception: pass
    return params

def contrastive_loss(model, prompt, target_new, target_true):
    tokens = model.to_tokens(prompt)
    new_id = model.to_tokens(target_new,  prepend_bos=False)[0,0]
    true_id= model.to_tokens(target_true, prepend_bos=False)[0,0]
    logits = model(tokens)
    lp = torch.nn.functional.log_softmax(logits[0,-1,:], dim=-1)
    loss = -lp[new_id] + lp[true_id]
    return loss, lp[new_id].exp().item(), lp[true_id].exp().item()

def measure(model, sample, ref_pack=None):
    """Eval-mode probe: returns (p_new, p_true, kl_to_ref or None)."""
    model.eval()
    with torch.no_grad():
        _, p_new, p_true = contrastive_loss(model, sample.prompt, sample.target_new, sample.target_true)
        kl = kl_loss_neutral_batched(model, *ref_pack, train=False).item() if ref_pack else None
    return p_new, p_true, kl

def edit_one(model, sample, attn, mlp, n_steps=5, lr=5e-3, beta_kl=0.1, grad_clip=1.0, ref_pack=None):
    params = get_circuit_params(model, attn, mlp)
    if not params:
        params = [p for layer in model.blocks for p in [layer.mlp.W_in, layer.mlp.W_out]]
    for p in model.parameters(): p.requires_grad_(False)
    for p in params: p.requires_grad_(True)
    opt = torch.optim.Adam(params, lr=lr)
    for _ in range(n_steps):
        model.train()
        opt.zero_grad(set_to_none=True)
        loss, _, _ = contrastive_loss(model, sample.prompt, sample.target_new, sample.target_true)
        if ref_pack is not None and beta_kl > 0:
            loss = loss + beta_kl * kl_loss_neutral_batched(model, *ref_pack, train=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, max_norm=grad_clip)
        opt.step()
        del loss
    for p in model.parameters(): p.requires_grad_(True)
    torch.cuda.empty_cache()

# Snapshot pristine weights once
print("Snapshotting original weights to CPU...")
original_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
torch.cuda.empty_cache()
free, _ = torch.cuda.mem_get_info()
print(f"GPU free after snapshot: {free/1e9:.1f} GB")

### 5.8 Sequential Edit Loop (v3 — protected second edit)

**Why v3 differs from v2:** v2 used the same edit protocol for both A and B, which caused B's gradient updates to completely overwrite A in nearly every pair (`retention_A ≈ 0.0` everywhere — no signal). The fix isn't to weaken B; it's to make the second edit faithful to the *reconsolidation* analogy: when the brain encodes a new memory, it actively protects the older one. We do this by adding a **second KL anchor** — KL to the post-A weights — during edit B. Now B has to satisfy two constraints: stay close to the original neutral set AND not destroy what A just learned.

**Protocol changes:**
1. **Dual-anchor KL during edit B**: `loss_B = contrastive_B + β1·KL_to_neutral + β2·KL_to_postA_at_A`
   - β1=0.1 (same as before, neutral-set anchor)
   - β2=0.3 (new, anchors edit B against post-A's distribution on the *A-prompt*)
2. **Edit B uses fewer steps (3 instead of 5)** — gives B less room to obliterate A
3. **Report raw `p_new_A_postAB`** alongside the ratio — so we see absolute values, not just normalized ones
4. **Save `rows` to disk after EVERY pair** — never lose data to a crash again

This is also a much more honest implementation of the reconsolidation framing from the proposal: *protein synthesis re-stabilizes the trace after recall*. In LLM terms: protect the just-edited circuit's outputs while modifying others.

**Metrics (unchanged + one addition):**
- `retention_A` = `p_new_A_postAB / p_new_A_postA` (1.0 = fully retained)
- `retention_A_abs` = `p_new_A_postAB` (raw — new in v3, lets us see if ratio is hiding signal)
- `success_B` = `p_new_B_postAB`
- `kl_super_additivity` = `kl_postAB - kl_A_solo - kl_B_solo`

In [ ]:
N_STEPS_A   = 5      # edit A: full strength (same as Notebook 3)
N_STEPS_B   = 3      # edit B: weaker (gives retention metric room to vary)
LR          = 5e-3
BETA_KL     = 0.1    # KL anchor to neutral set (both edits)
BETA_PROTECT= 0.3    # additional KL anchor protecting A during edit B
GRAD_CLIP   = 1.0

method_circuits = {
    "ACDC":      acdc_by_cf_idx,
    "ROMEtrace": rome_by_cf_idx,
    "Random":    random_by_cf_idx,
}

all_pairs = ([("shared",   a, b) for a, b in share_pairs]
          + [("disjoint", a, b) for a, b in disjoint_pairs])
print(f"Total pair·method runs to execute: {len(all_pairs)} pairs × {len(method_circuits)} methods = {len(all_pairs)*len(method_circuits)}")

# === Crash-safe save: write rows to disk after every pair ===
ts_run = datetime.now().strftime("%Y%m%d_%H%M%S")
ROWS_FILE = RESULTS_DIR / f"rows_in_progress_{ts_run}.json"
print(f"Crash-safe save target: {ROWS_FILE}")

def save_rows(rows):
    """Atomic write — temp file then rename, so a crash mid-write can't corrupt."""
    tmp = ROWS_FILE.with_suffix(".tmp")
    with open(tmp, "w") as f:
        json.dump({"rows": rows, "config": {
            "n_steps_A": N_STEPS_A, "n_steps_B": N_STEPS_B,
            "lr": LR, "beta_kl": BETA_KL, "beta_protect": BETA_PROTECT,
            "grad_clip": GRAD_CLIP,
        }}, f, indent=2)
    tmp.replace(ROWS_FILE)

# === Edit B with dual KL anchor: neutral-set + protect-A ===
def edit_B_protected(model, sample_B, attn_B, mlp_B, A_prompt, A_postA_lp,
                     n_steps, lr, beta_kl, beta_protect, grad_clip, ref_pack):
    """
    Edit B while protecting A.
    A_postA_lp: log-probs at A_prompt's last token RIGHT AFTER edit A finished (cached once).
    We add KL(A_postA_lp || edit_lp_at_A_prompt) to the loss — penalises drift on A's prompt.
    """
    params = get_circuit_params(model, attn_B, mlp_B)
    if not params:
        params = [p for layer in model.blocks for p in [layer.mlp.W_in, layer.mlp.W_out]]
    for p in model.parameters(): p.requires_grad_(False)
    for p in params: p.requires_grad_(True)
    opt = torch.optim.Adam(params, lr=lr)

    A_tokens = model.to_tokens(A_prompt)

    for _ in range(n_steps):
        model.train()
        opt.zero_grad(set_to_none=True)
        loss, _, _ = contrastive_loss(model, sample_B.prompt, sample_B.target_new, sample_B.target_true)
        # KL on neutral set
        if ref_pack is not None and beta_kl > 0:
            loss = loss + beta_kl * kl_loss_neutral_batched(model, *ref_pack, train=True)
        # KL protecting A's distribution (the new, reconsolidation-faithful constraint)
        if beta_protect > 0:
            logits_at_A = model(A_tokens)
            edit_lp_at_A = torch.nn.functional.log_softmax(logits_at_A[0,-1,:], dim=-1)
            p_postA = A_postA_lp.exp()
            kl_protect = (p_postA * (A_postA_lp - edit_lp_at_A)).sum()
            loss = loss + beta_protect * kl_protect
            del logits_at_A
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, max_norm=grad_clip)
        opt.step()
        del loss
    for p in model.parameters(): p.requires_grad_(True)
    torch.cuda.empty_cache()

def cache_logp_at_prompt(model, prompt):
    """Cache log-probs at last token of prompt — used as KL reference for protection."""
    model.eval()
    with torch.no_grad():
        logits = model(model.to_tokens(prompt))
        lp = torch.nn.functional.log_softmax(logits[0,-1,:], dim=-1).detach().clone()
    return lp

rows = []
started = datetime.now()

for pair_idx, (cond, A, B) in enumerate(all_pairs):
    print(f"\n--- Pair {pair_idx+1}/{len(all_pairs)} ({cond}) :: A=[{A.idx_cf}] B=[{B.idx_cf}] ---")
    restore_weights(model, original_state)
    ref_pack = cache_reference_logits_batched(model, NEUTRAL_PROMPTS)

    for method_name, circ_map in method_circuits.items():
        cA = circ_map.get(A.idx_cf); cB = circ_map.get(B.idx_cf)
        if cA is None or cB is None:
            print(f"  {method_name}: SKIP (missing circuit)")
            continue
        try:
            # ---- Solo reference for A ----
            restore_weights(model, original_state)
            edit_one(model, A, cA.get("attn_heads", []), cA["mlp_layers"],
                     n_steps=N_STEPS_A, lr=LR, beta_kl=BETA_KL, grad_clip=GRAD_CLIP, ref_pack=ref_pack)
            p_new_A_solo, p_true_A_solo, kl_A_solo = measure(model, A, ref_pack=ref_pack)
            # ---- Solo reference for B (use same N_STEPS_B as sequential B for fair comparison) ----
            restore_weights(model, original_state)
            edit_one(model, B, cB.get("attn_heads", []), cB["mlp_layers"],
                     n_steps=N_STEPS_B, lr=LR, beta_kl=BETA_KL, grad_clip=GRAD_CLIP, ref_pack=ref_pack)
            p_new_B_solo, p_true_B_solo, kl_B_solo = measure(model, B, ref_pack=ref_pack)
            # ---- Sequential A then B with PROTECTION ----
            restore_weights(model, original_state)
            edit_one(model, A, cA.get("attn_heads", []), cA["mlp_layers"],
                     n_steps=N_STEPS_A, lr=LR, beta_kl=BETA_KL, grad_clip=GRAD_CLIP, ref_pack=ref_pack)
            p_new_A_postA, _, kl_postA = measure(model, A, ref_pack=ref_pack)
            # cache A's distribution AT A's prompt — this is what we protect
            A_postA_lp = cache_logp_at_prompt(model, A.prompt)
            # edit B with dual-anchor KL
            edit_B_protected(model, B, cB.get("attn_heads", []), cB["mlp_layers"],
                             A_prompt=A.prompt, A_postA_lp=A_postA_lp,
                             n_steps=N_STEPS_B, lr=LR,
                             beta_kl=BETA_KL, beta_protect=BETA_PROTECT,
                             grad_clip=GRAD_CLIP, ref_pack=ref_pack)
            p_new_A_postAB, p_true_A_postAB, kl_postAB = measure(model, A, ref_pack=ref_pack)
            p_new_B_postAB, p_true_B_postAB, _ = measure(model, B, ref_pack=ref_pack)

            retention_A     = p_new_A_postAB / max(p_new_A_postA, 1e-8)
            retention_A_abs = p_new_A_postAB   # raw — new in v3
            success_B       = p_new_B_postAB
            kl_super        = kl_postAB - (kl_A_solo + kl_B_solo)

            row = {
                "pair_idx": pair_idx, "condition": cond, "method": method_name,
                "A_idx": A.idx_cf, "B_idx": B.idx_cf,
                "A_subject": A.subject, "B_subject": B.subject,
                "A_relation": A.relation_id, "B_relation": B.relation_id,
                "n_mlp_A": cA["n_mlp"], "n_mlp_B": cB["n_mlp"],
                "n_attn_A": cA.get("n_attn", 0), "n_attn_B": cB.get("n_attn", 0),
                "p_new_A_solo": p_new_A_solo, "p_new_B_solo": p_new_B_solo,
                "p_new_A_postA": p_new_A_postA, "p_new_A_postAB": p_new_A_postAB,
                "p_new_B_postAB": p_new_B_postAB,
                "p_true_A_postAB": p_true_A_postAB, "p_true_B_postAB": p_true_B_postAB,
                "kl_A_solo": kl_A_solo, "kl_B_solo": kl_B_solo,
                "kl_postA": kl_postA, "kl_postAB": kl_postAB,
                "retention_A": retention_A,
                "retention_A_abs": retention_A_abs,
                "success_B": success_B,
                "kl_super_additivity": kl_super,
                "status": "ok",
            }
            rows.append(row)
            print(f"  {method_name:11s}  retA={retention_A:.3f} (abs={retention_A_abs:.3f}) succB={success_B:.3f}  kl_super={kl_super:+.2f}")
        except Exception as e:
            print(f"  {method_name:11s}  FAILED: {type(e).__name__}: {str(e)[:100]}")
            rows.append({
                "pair_idx": pair_idx, "condition": cond, "method": method_name,
                "A_idx": A.idx_cf, "B_idx": B.idx_cf,
                "status": "failed", "error": f"{type(e).__name__}: {str(e)[:200]}",
            })
            torch.cuda.empty_cache()

    # ---- Save after every pair (crash-safe) ----
    save_rows(rows)

elapsed = (datetime.now() - started).total_seconds() / 60
print(f"\nTotal runtime: {elapsed:.1f} min  ({len(rows)} rows logged)")
print(f"Crash-safe data: {ROWS_FILE}")
restore_weights(model, original_state)


### 5.9 Aggregate and Save

In [ ]:
ok_rows = [r for r in rows if r.get("status") == "ok"]
print(f"OK rows: {len(ok_rows)}/{len(rows)}")

agg = defaultdict(lambda: defaultdict(list))
for r in ok_rows:
    key = (r["condition"], r["method"])
    agg[key]["retention_A"].append(r["retention_A"])
    agg[key]["retention_A_abs"].append(r["retention_A_abs"])
    agg[key]["success_B"].append(r["success_B"])
    agg[key]["kl_super_additivity"].append(r["kl_super_additivity"])

summary = {}
print(f"\n{'cond':>10} {'method':>11}  {'retA_ratio':>11} {'retA_abs':>10} {'succB':>8} {'kl_super':>10}  n")
print("-" * 80)
for cond in ["shared", "disjoint"]:
    for method in ["ACDC", "ROMEtrace", "Random"]:
        key = (cond, method)
        if not agg[key]["retention_A"]: continue
        rA = np.array(agg[key]["retention_A"])
        rAa= np.array(agg[key]["retention_A_abs"])
        sB = np.array(agg[key]["success_B"])
        ks = np.array(agg[key]["kl_super_additivity"])
        summary[f"{cond}__{method}"] = {
            "n": int(len(rA)),
            "retention_A":     {"mean": float(rA.mean()), "std": float(rA.std())},
            "retention_A_abs": {"mean": float(rAa.mean()), "std": float(rAa.std())},
            "success_B":       {"mean": float(sB.mean()), "std": float(sB.std())},
            "kl_super_additivity": {"mean": float(ks.mean()), "std": float(ks.std())},
        }
        print(f"{cond:>10} {method:>11}  {rA.mean():>11.3f} {rAa.mean():>10.3f} {sB.mean():>8.3f} {ks.mean():>+10.2f}  {len(rA)}")

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out = {
    "author": "Amogh", "notebook": "Notebook 5 v3 — Sequential Editing (protected B)",
    "model": MODEL_NAME, "timestamp": ts,
    "n_pairs": {"shared": len(share_pairs), "disjoint": len(disjoint_pairs)},
    "config": {"n_steps_A": N_STEPS_A, "n_steps_B": N_STEPS_B,
               "beta_kl": BETA_KL, "beta_protect": BETA_PROTECT,
               "grad_clip": GRAD_CLIP, "lr": LR},
    "summary": summary,
}
with open(RESULTS_DIR / f"summary_nb5v3_{ts}.json", "w") as f:
    json.dump(out, f, indent=2)
with open(RESULTS_DIR / f"sequential_edit_rows_v3_{ts}.json", "w") as f:
    json.dump({"rows": rows}, f, indent=2)
print(f"\nSaved -> {RESULTS_DIR}/summary_nb5v3_{ts}.json")
print(f"Saved -> {RESULTS_DIR}/sequential_edit_rows_v3_{ts}.json")

### 5.10 Statistical Tests

Two pre-registered hypotheses:

**H1 (interaction):** retention_A is *lower* in `shared` than `disjoint` — collapsed across methods.
**H2 (method × condition):** within `shared`, retention_A differs across methods (test ACDC vs ROMEtrace vs Random).

We use Mann-Whitney U for H1 (paired-by-pair_idx is impossible across pair sets, so unpaired) and Kruskal-Wallis for H2 (small n, non-parametric).

In [ ]:
from scipy import stats

# H1
shared_ret   = [r["retention_A"] for r in ok_rows if r["condition"] == "shared"]
disjoint_ret = [r["retention_A"] for r in ok_rows if r["condition"] == "disjoint"]
if shared_ret and disjoint_ret:
    u, p = stats.mannwhitneyu(shared_ret, disjoint_ret, alternative="less")
    print(f"H1: retention_A(shared) < retention_A(disjoint)?")
    print(f"    Mann-Whitney U = {u:.1f}, one-sided p = {p:.4f}")
    print(f"    means: shared={np.mean(shared_ret):.3f}  disjoint={np.mean(disjoint_ret):.3f}")

# H2 — within shared, by method
groups = {m: [r["retention_A"] for r in ok_rows if r["condition"]=="shared" and r["method"]==m] for m in ["ACDC","ROMEtrace","Random"]}
groups = {k: v for k, v in groups.items() if v}
if len(groups) >= 2:
    h, p = stats.kruskal(*groups.values())
    print(f"\nH2: retention_A differs across methods within `shared`?")
    print(f"    Kruskal-Wallis H = {h:.2f}, p = {p:.4f}")
    for m, vs in groups.items():
        print(f"    {m:>11}: mean={np.mean(vs):.3f}, n={len(vs)}")

# Same tests on KL super-additivity
shared_ks   = [r["kl_super_additivity"] for r in ok_rows if r["condition"] == "shared"]
disjoint_ks = [r["kl_super_additivity"] for r in ok_rows if r["condition"] == "disjoint"]
if shared_ks and disjoint_ks:
    u, p = stats.mannwhitneyu(shared_ks, disjoint_ks, alternative="greater")
    print(f"\nH3: kl_super_additivity(shared) > kl_super_additivity(disjoint)?")
    print(f"    Mann-Whitney U = {u:.1f}, one-sided p = {p:.4f}")
    print(f"    means: shared={np.mean(shared_ks):+.2f}  disjoint={np.mean(disjoint_ks):+.2f}")

### 5.11 Figures

In [ ]:
FIG_DIR = RESULTS_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

methods_order = ["ACDC", "ROMEtrace", "Random"]
conds = ["shared", "disjoint"]
colors = {"shared": "#cc3333", "disjoint": "#2266aa"}

# Figure 1 — Retention of A: condition × method grouped bars
fig, ax = plt.subplots(figsize=(7, 4.5))
x = np.arange(len(methods_order))
w = 0.36
for ci, cond in enumerate(conds):
    means = [np.mean([r["retention_A"] for r in ok_rows if r["method"]==m and r["condition"]==cond]) if any(r["method"]==m and r["condition"]==cond for r in ok_rows) else 0 for m in methods_order]
    sds   = [np.std ([r["retention_A"] for r in ok_rows if r["method"]==m and r["condition"]==cond]) if any(r["method"]==m and r["condition"]==cond for r in ok_rows) else 0 for m in methods_order]
    ax.bar(x + (ci-0.5)*w, means, w, yerr=sds, label=cond, color=colors[cond], capsize=3, alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(methods_order)
ax.set_ylabel("Retention of A after editing B  (p_new_A_postAB / p_new_A_postA)")
ax.set_title("Sequential edit interference by localization × subject overlap")
ax.axhline(1.0, color="gray", lw=0.8, ls="--", alpha=0.6); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIG_DIR / "fig1_retention.png", dpi=140); plt.show()

# Figure 2 — KL super-additivity (the headline if H3 fires)
fig, ax = plt.subplots(figsize=(7, 4.5))
for ci, cond in enumerate(conds):
    means = [np.mean([r["kl_super_additivity"] for r in ok_rows if r["method"]==m and r["condition"]==cond]) if any(r["method"]==m and r["condition"]==cond for r in ok_rows) else 0 for m in methods_order]
    sds   = [np.std ([r["kl_super_additivity"] for r in ok_rows if r["method"]==m and r["condition"]==cond]) if any(r["method"]==m and r["condition"]==cond for r in ok_rows) else 0 for m in methods_order]
    ax.bar(x + (ci-0.5)*w, means, w, yerr=sds, label=cond, color=colors[cond], capsize=3, alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(methods_order)
ax.set_ylabel("KL super-additivity  (kl_postAB − kl_A_solo − kl_B_solo)")
ax.set_title("Do edits compound or cancel? Positive = compound, Zero = independent")
ax.axhline(0.0, color="black", lw=0.8); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIG_DIR / "fig2_kl_super_additivity.png", dpi=140); plt.show()

# Figure 3 — Per-pair retention scatter (most useful for spotting per-pair patterns)
fig, ax = plt.subplots(figsize=(7, 4.5))
marker = {"ACDC": "o", "ROMEtrace": "s", "Random": "^"}
for cond in conds:
    for m in methods_order:
        rs = [r for r in ok_rows if r["method"]==m and r["condition"]==cond]
        if not rs: continue
        ax.scatter([r["retention_A"] for r in rs], [r["success_B"] for r in rs],
                   marker=marker[m], color=colors[cond], alpha=0.65, s=55,
                   label=f"{cond} / {m}", edgecolors="black", linewidths=0.4)
ax.set_xlabel("Retention of A (1.0 = fully retained)")
ax.set_ylabel("Success of B (p_new_B after sequential edit)")
ax.set_title("Per-pair retention vs. follow-on edit success")
ax.axvline(1.0, color="gray", lw=0.5, ls=":"); ax.axhline(0.5, color="gray", lw=0.5, ls=":")
ax.legend(fontsize=8, ncol=2, loc="lower left"); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIG_DIR / "fig3_pareto_per_pair.png", dpi=140); plt.show()

print(f"\nFigures saved to {FIG_DIR}/")

### 5.12 Download Results

Pull everything down for the writeup.

In [ ]:
import shutil
ts2 = datetime.now().strftime("%Y%m%d_%H%M%S")
bundle = f"nb5_results_{ts2}"
shutil.make_archive(bundle, "zip", RESULTS_DIR)
print(f"Bundled: {bundle}.zip")
from google.colab import files
files.download(f"{bundle}.zip")

### What to look for in the results

**Strong novel finding** if you see:
- `retention_A` in `shared` is significantly < `retention_A` in `disjoint` (H1)
- AND the gap is *larger* for tighter circuits (ROMEtrace tighter than Random) within `shared` (H2)
- AND `kl_super_additivity` is positive in `shared`, near zero in `disjoint` (H3)

**Headline:** *"Localization tightness amplifies interference under sequential edits with shared subjects — exactly the regime memory reconsolidation predicts."* This connects back to your reconsolidation framing in a non-trivial way: in the brain, repeated retrieval of related memories is when interference effects show up; we'd be observing the same phenomenon in LLM weight edits.

**Weaker (but still novel) finding** if H1 holds but H2 doesn't: localization choice doesn't matter for interference, but subject overlap does. Still a contribution to the editing-stability literature.

**Null result** if neither holds: you can still report it as "under KL-stabilized gradient editing with our protocol, sequential interference is bounded by the KL constraint regardless of localization or subject relationship." This is publishable as a methodology note — KL regularization is doing more work than localization, and is sufficient on its own.

Either way, the single experiment lands a defensible finding.